In [1]:
# =========================================================
# 02_clean_dataset.ipynb / clean_dataset.py
# Spotify Popularity Project — Cleaning Pipeline
# =========================================================

import pandas as pd
import numpy as np


In [2]:
# =========================================================
# LOAD DATASET
# =========================================================

FILE_PATH = "../raw_data/spotify-tracks-dataset.csv"

print("Loading dataset...")

df = pd.read_csv(FILE_PATH)

print("Dataset loaded.")
print("Original shape:", df.shape)


Loading dataset...
Dataset loaded.
Original shape: (550622, 23)


In [3]:
# =========================================================
# COLUMN NAMES
# =========================================================

TRACK_COL = "name"
ARTIST_COL = "artists"
YEAR_COL = "year"
GENRE_COL = "genre"
POPULARITY_COL = "popularity"
ARTIST_POP_COL = "avg_artist_popularity"

# Audio feature columns
AUDIO_FEATURES = [
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
]

In [4]:
# =========================================================
# REMOVE POPULARITY = 0
# =========================================================

print("\nRemoving popularity = 0 tracks...")

df = df[df[POPULARITY_COL] > 0]

print("Shape after removing zeros:", df.shape)



Removing popularity = 0 tracks...
Shape after removing zeros: (401844, 23)


In [5]:
# =========================================================
# REMOVE DUPLICATES
# Keep highest popularity version
# =========================================================

print("\nRemoving duplicate tracks...")

# Create normalized track-artist key
df["_track_artist"] = (
    df[TRACK_COL]
    .astype(str)
    .str.lower()
    .str.strip()
    +
    " - " +
    df[ARTIST_COL]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Sort by popularity descending
df = df.sort_values(
    by=POPULARITY_COL,
    ascending=False
)

# Keep highest popularity duplicate
df = df.drop_duplicates(
    subset="_track_artist",
    keep="first"
)

print("Shape after deduplication:", df.shape)


Removing duplicate tracks...
Shape after deduplication: (363142, 24)


In [6]:
# =========================================================
# CREATE ERA BUCKETS
# =========================================================

print("\nCreating era buckets...")

def year_to_era(year):

    if year < 1980:
        return "classic"

    elif year < 2000:
        return "retro"

    elif year < 2010:
        return "early_streaming"

    else:
        return "modern"

df["era"] = df[YEAR_COL].apply(year_to_era)

print(df["era"].value_counts())



Creating era buckets...
era
modern             185046
early_streaming    108074
retro               44052
classic             25970
Name: count, dtype: int64


In [7]:
# =========================================================
# ONE-HOT ENCODE GENRE
# =========================================================

print("\nEncoding genres...")

df = pd.get_dummies(
    df,
    columns=[GENRE_COL],
    prefix="genre"
)


Encoding genres...


In [8]:
# =========================================================
# DROP TEMP COLUMN
# =========================================================

df = df.drop(columns=["_track_artist"])

In [9]:
# =========================================================
# FINAL CHECKS
# =========================================================

print("\nFinal dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nPopularity statistics:")
print(df[POPULARITY_COL].describe())


Final dataset shape: (363142, 33)

Missing values:
album_name      13
name             1
id               0
artists          0
danceability     0
energy           0
key              0
loudness         0
mode             0
speechiness      0
dtype: int64

Popularity statistics:
count    363142.000000
mean         24.552638
std          16.132035
min           1.000000
25%          12.000000
50%          23.000000
75%          35.000000
max          98.000000
Name: popularity, dtype: float64
